# 02 — Preprocessing

Data cleaning, feature engineering, normalization/scaling, and the chronological train/test split logic used by all three models.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data_loader import load_raw_data, save_processed_data
from src.preprocessor import build_feature_pipeline, chronological_split, SeriesScaler
from src.utils import load_config

cfg = load_config('../config.yaml')
data_cfg, prep_cfg = cfg['data'], cfg['preprocessing']
target = data_cfg['target_column']

## 1. Load raw data

In [ ]:
raw_df = load_raw_data('../' + data_cfg['raw_path'], data_cfg['datetime_column'], data_cfg['frequency'])
raw_df.isna().sum()

## 2. Cleaning + feature engineering

Missing-value interpolation, calendar features, lag features, and rolling statistics — see `src/preprocessor.py`.

In [ ]:
features_df = build_feature_pipeline(
    raw_df,
    target,
    prep_cfg['fill_method'],
    prep_cfg['lag_features'],
    prep_cfg['rolling_windows'],
    prep_cfg['add_time_features'],
)
features_df.head()

## 3. Chronological train / validation / test split

Time series must never be split randomly — a chronological split preserves temporal order and avoids leakage from the future into training.

In [ ]:
train_df, val_df, test_df = chronological_split(
    features_df, data_cfg['test_size'], data_cfg['validation_size']
)
print(len(train_df), len(val_df), len(test_df))

## 4. Normalization / scaling

Fit the scaler on the training split only, then apply it to validation/test to avoid data leakage. Required for the LSTM model; tree-based/statistical models use the unscaled features.

In [ ]:
scaler = SeriesScaler(prep_cfg['scaling_method'])
train_scaled = scaler.fit_transform(train_df[target])
test_scaled = scaler.transform(test_df[target])
train_scaled[:5], test_scaled[:5]

## 5. Persist processed data

In [ ]:
save_processed_data(features_df, '../' + data_cfg['processed_path'])